# TheStatsAPI league coverage audit

This notebook screens Greece, the Netherlands, Turkey, Belgium, Portugal, and Scotland for seasons **2020-21 through 2025-26**. It does not download the full player dataset.

The audit uses every fixture for fixture coverage and Football-Data joining, but only five deterministic matches per league-season for expensive lineup and player-stat checks. Successful API responses are cached locally, so rerunning the notebook does not spend requests twice.

Set `RUN_AUDIT = True` only when ready. The notebook has a hard 750-request ceiling for this run, preserving most of the 10,000-request trial. Sample-based checks are provisional; a passing league is a candidate for full collection, not a final guarantee.

In [ ]:
import hashlib
import json
import os
import re
import sqlite3
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["THESTATSAPI_KEY"]
BASE_URL = "https://api.thestatsapi.com/api"
ODDS_DB = Path("odds.db")
CACHE_DIR = Path("data/statsapi_audit_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Safety switch: inspect the estimated request count before changing this.
RUN_AUDIT = True
REQUEST_BUDGET = 750
REQUEST_SLEEP_SEC = 2.5
MAX_429_RETRIES = 5
SAMPLE_MATCHES_PER_SEASON = 5
SEASON_START_YEARS = list(range(2020, 2026))

LEAGUES = {
    "greece": {"competition_id": "comp_4008", "division": "G1"},
    "netherlands": {"competition_id": "comp_3809", "division": "N1"},
    "turkey": {"competition_id": "comp_9235", "division": "T1"},
    "belgium": {"competition_id": "comp_8531", "division": "B1"},
    "portugal": {"competition_id": "comp_8385", "division": "P1"},
    "scotland": {"competition_id": "comp_6387", "division": "SC0"},
}

THRESHOLDS = {
    "fixture_coverage": 0.95,
    "both_lineups": 0.90,
    "twenty_two_starters": 0.90,
    "player_stats_matches": 0.90,
    "starter_minutes": 0.90,
    "player_identity": 0.99,
    "football_data_join": 0.95,
}

estimated_detail_requests = (
    len(LEAGUES) * len(SEASON_START_YEARS) * SAMPLE_MATCHES_PER_SEASON * 2
)
estimated_list_requests = len(LEAGUES) + len(LEAGUES) * len(SEASON_START_YEARS) * 4
print("Conservative first-run estimate:", estimated_detail_requests + estimated_list_requests)
print("Hard request budget:", REQUEST_BUDGET)
print("Cached responses make later reruns cheaper.")

Conservative first-run estimate: 510
Hard request budget: 750
Cached responses make later reruns cheaper.


In [32]:
SESSION = requests.Session()
SESSION.headers.update({"Authorization": f"Bearer {API_KEY}"})
REQUESTS_USED_THIS_RUN = 0
CACHE_HITS_THIS_RUN = 0


class ApiError(RuntimeError):
    pass


def _cache_path(path: str, params: dict | None) -> Path:
    raw = json.dumps([path, params or {}], sort_keys=True, default=str).encode()
    return CACHE_DIR / f"{hashlib.sha256(raw).hexdigest()}.json"


def api_get(path: str, params: dict | None = None, *, use_cache: bool = True) -> dict:
    global REQUESTS_USED_THIS_RUN, CACHE_HITS_THIS_RUN
    cache_path = _cache_path(path, params)
    if use_cache and cache_path.exists():
        CACHE_HITS_THIS_RUN += 1
        return json.loads(cache_path.read_text(encoding="utf-8"))

    url = f"{BASE_URL}{path}"
    for attempt in range(MAX_429_RETRIES + 1):
        if REQUESTS_USED_THIS_RUN >= REQUEST_BUDGET:
            raise ApiError(f"Stopped at the {REQUEST_BUDGET}-request notebook budget")
        if REQUESTS_USED_THIS_RUN:
            time.sleep(REQUEST_SLEEP_SEC)
        response = SESSION.get(url, params=params or {}, timeout=30)
        REQUESTS_USED_THIS_RUN += 1

        if response.status_code == 429 and attempt < MAX_429_RETRIES:
            retry_after = response.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else min(60, 5 * (2**attempt))
            print(f"Rate limited; waiting {wait:.0f}s before retrying")
            time.sleep(wait)
            continue

        if not response.ok:
            try:
                error = response.json().get("error", {})
                message = error.get("message", response.text)
            except Exception:
                message = response.text
            raise ApiError(f"{response.status_code} {path}: {message}")

        payload = response.json()
        if use_cache:
            cache_path.write_text(json.dumps(payload), encoding="utf-8")
        return payload

    raise ApiError(f"Repeated rate limits for {path}")


def api_paginate(path: str, params: dict | None = None) -> list[dict]:
    params = dict(params or {})
    params.setdefault("per_page", 100)
    rows = []
    page = 1
    while True:
        payload = api_get(path, {**params, "page": page})
        rows.extend(payload.get("data", []))
        total_pages = payload.get("meta", {}).get("total_pages", page)
        if page >= total_pages:
            return rows
        page += 1

In [33]:
# Known provider-name differences. Add aliases only when the unmatched-team table shows a real mismatch.
TEAM_ALIASES = {
    "aekathens": "aek",
    "asterasaktor": "asterastripolis",
    "atromitosathens": "atromitos",
    "aristhessaloniki": "aris",
    "athenskallitheafc": "athenskallithea",
    "mgspanserraikos": "panserraikos",
}


def normalize_team_name(name: str) -> str:
    value = (name or "").lower().strip()
    value = re.sub(r"\b(fc|cf|sc|afc|fk|sk)\b", "", value)
    value = re.sub(r"[^a-z0-9]", "", value)
    return TEAM_ALIASES.get(value, value)


def season_label(start_year: int) -> str:
    return f"{start_year}-{(start_year + 1) % 100:02d}"


def load_football_data(division: str, start_year: int) -> pd.DataFrame:
    query = """
        SELECT match_date, home_team, away_team, result_3way, odds_is_closing
        FROM historical_results_odds
        WHERE division = ? AND season = ?
    """
    with sqlite3.connect(ODDS_DB) as connection:
        frame = pd.read_sql_query(query, connection, params=[division, season_label(start_year)])
    frame["fd_key"] = (
        frame["match_date"].astype(str)
        + "|" + frame["home_team"].map(normalize_team_name)
        + "|" + frame["away_team"].map(normalize_team_name)
    )
    return frame

In [34]:
def find_season_id(competition_id: str, start_year: int) -> str | None:
    seasons = api_paginate(f"/football/competitions/{competition_id}/seasons")
    match = next(
        (season for season in seasons
         if season.get("start_year") == start_year
         and season.get("end_year") == start_year + 1),
        None,
    )
    return None if match is None else match["id"]


def load_statsapi_matches(competition_id: str, season_id: str) -> pd.DataFrame:
    matches = api_paginate(
        "/football/matches",
        {"competition_id": competition_id, "season_id": season_id, "status": "finished"},
    )
    if not matches:
        return pd.DataFrame()
    frame = pd.DataFrame(matches)
    frame["utc_date"] = pd.to_datetime(frame["utc_date"], utc=True)
    frame["match_date"] = frame["utc_date"].dt.strftime("%Y-%m-%d")
    frame["home_name"] = frame["home_team"].map(lambda value: value.get("name"))
    frame["away_name"] = frame["away_team"].map(lambda value: value.get("name"))
    frame["stats_key"] = (
        frame["match_date"]
        + "|" + frame["home_name"].map(normalize_team_name)
        + "|" + frame["away_name"].map(normalize_team_name)
    )
    return frame.sort_values(["utc_date", "id"]).reset_index(drop=True)


def deterministic_sample(frame: pd.DataFrame, size: int) -> pd.DataFrame:
    if len(frame) <= size:
        return frame.copy()
    indexes = [round(i * (len(frame) - 1) / (size - 1)) for i in range(size)]
    return frame.iloc[indexes].copy()

In [35]:
def _player_id(record: dict) -> Any:
    nested = record.get("player") or {}
    return record.get("player_id") or record.get("id") or nested.get("id")


def _player_name(record: dict) -> str | None:
    nested = record.get("player") or {}
    return record.get("player_name") or record.get("name") or nested.get("name")


def _minutes(record: dict) -> Any:
    return record.get("minutes", record.get("minutes_played"))


def fetch_optional(path: str, params: dict | None = None) -> tuple[Any, str | None]:
    try:
        return api_get(path, params).get("data"), None
    except ApiError as error:
        return None, str(error)


def audit_match(row: pd.Series) -> dict:
    match_id = row["id"]
    lineups, lineup_error = fetch_optional(f"/football/matches/{match_id}/lineups")
    # The public API documentation exposes per-match player rows through this endpoint/filter.
    stats, stats_error = fetch_optional("/football/players/stats", {"match_id": match_id, "per_page": 100})
    lineups = lineups or {}
    stats = stats or []

    home_xi = (lineups.get("home") or {}).get("starting_xi") or []
    away_xi = (lineups.get("away") or {}).get("starting_xi") or []
    starters = home_xi + away_xi
    starter_ids = [_player_id(player) for player in starters if _player_id(player)]
    stats_by_player = {_player_id(player): player for player in stats if _player_id(player)}
    starters_with_minutes = sum(
        1 for player_id in starter_ids
        if player_id in stats_by_player and _minutes(stats_by_player[player_id]) is not None
    )

    identity_records = list(starters) + list(stats)
    identity_with_id = sum(bool(_player_id(player)) for player in identity_records)
    stat_fields = sorted({key for player in stats for key in player.keys()})

    return {
        "match_id": match_id,
        "match_date": row["match_date"],
        "home_team": row["home_name"],
        "away_team": row["away_name"],
        "both_lineups": bool(home_xi) and bool(away_xi),
        "starter_count": len(starters),
        "twenty_two_starters": len(starters) == 22,
        "player_stats_available": bool(stats),
        "starters_with_minutes": starters_with_minutes,
        "starters_with_id": len(starter_ids),
        "identity_rows": len(identity_records),
        "identity_rows_with_id": identity_with_id,
        "player_stat_fields": stat_fields,
        "lineup_error": lineup_error,
        "stats_error": stats_error,
    }

In [36]:
audit_rows = []
detail_rows = []
unmatched_rows = []

if not RUN_AUDIT:
    print("Dry run only. Review the configuration, then set RUN_AUDIT = True and rerun from the top.")
else:
    for league, config in LEAGUES.items():
        print(f"\n--- {league.upper()} ---")
        for start_year in SEASON_START_YEARS:
            label = season_label(start_year)
            season_id = find_season_id(config["competition_id"], start_year)
            fd = load_football_data(config["division"], start_year)

            if season_id is None:
                audit_rows.append({
                    "league": league, "season": label, "season_found": False,
                    "fd_fixtures": len(fd), "api_fixtures": 0,
                })
                print(label, "season unavailable")
                continue

            matches = load_statsapi_matches(config["competition_id"], season_id)
            if matches.empty:
                audit_rows.append({
                    "league": league, "season": label, "season_found": True,
                    "fd_fixtures": len(fd), "api_fixtures": 0,
                })
                print(label, "no finished matches")
                continue

            joined = matches.merge(fd[["fd_key"]], left_on="stats_key", right_on="fd_key", how="left")
            unmatched = joined[joined["fd_key"].isna()][["match_date", "home_name", "away_name"]].copy()
            if not unmatched.empty:
                unmatched.insert(0, "season", label)
                unmatched.insert(0, "league", league)
                unmatched_rows.extend(unmatched.to_dict("records"))

            sample = deterministic_sample(matches, SAMPLE_MATCHES_PER_SEASON)
            season_details = []
            for _, match in sample.iterrows():
                detail = audit_match(match)
                detail.update({"league": league, "season": label})
                season_details.append(detail)
                detail_rows.append(detail)

            details = pd.DataFrame(season_details)
            total_starters = int(details["starters_with_id"].sum())
            total_identity_rows = int(details["identity_rows"].sum())
            xg_rate = float(matches.get("xg_available", pd.Series(False, index=matches.index)).fillna(False).mean())
            audit_rows.append({
                "league": league,
                "season": label,
                "season_found": True,
                "fd_fixtures": len(fd),
                "api_fixtures": len(matches),
                "fixture_coverage": len(matches) / len(fd) if len(fd) else None,
                "football_data_join": joined["fd_key"].notna().mean(),
                "sample_matches": len(details),
                "both_lineups": details["both_lineups"].mean(),
                "twenty_two_starters": details["twenty_two_starters"].mean(),
                "player_stats_matches": details["player_stats_available"].mean(),
                "starter_minutes": details["starters_with_minutes"].sum() / total_starters if total_starters else 0.0,
                "player_identity": details["identity_rows_with_id"].sum() / total_identity_rows if total_identity_rows else 0.0,
                "xg_flag_rate": xg_rate,
            })
            print(label, f"fixtures={len(matches)}, detailed_sample={len(details)}")

    print(f"\nRequests used this run: {REQUESTS_USED_THIS_RUN}")
    print(f"Cache hits this run: {CACHE_HITS_THIS_RUN}")


--- GREECE ---
2020-21 fixtures=239, detailed_sample=5
Rate limited; waiting 26s before retrying
2021-22 fixtures=242, detailed_sample=5
Rate limited; waiting 23s before retrying
2022-23 fixtures=240, detailed_sample=5
Rate limited; waiting 23s before retrying
2023-24 fixtures=240, detailed_sample=5
Rate limited; waiting 24s before retrying
2024-25 fixtures=236, detailed_sample=5
Rate limited; waiting 23s before retrying
2025-26 fixtures=236, detailed_sample=5

--- NETHERLANDS ---
Rate limited; waiting 23s before retrying
2020-21 fixtures=286, detailed_sample=5
Rate limited; waiting 24s before retrying
Rate limited; waiting 23s before retrying
2021-22 fixtures=302, detailed_sample=5
Rate limited; waiting 23s before retrying
2022-23 fixtures=306, detailed_sample=5
Rate limited; waiting 24s before retrying
2023-24 fixtures=321, detailed_sample=5
Rate limited; waiting 23s before retrying
2024-25 fixtures=321, detailed_sample=5
Rate limited; waiting 24s before retrying
2025-26 fixtures=32

ApiError: 404 /football/competitions/comp_8532/seasons: Competition not found

In [28]:
if not audit_rows:
    print("No results yet. Run the previous cell with RUN_AUDIT = True.")
else:
    audit_summary = pd.DataFrame(audit_rows)
    metric_columns = list(THRESHOLDS)
    for metric in metric_columns:
        audit_summary[f"{metric}_pass"] = audit_summary.get(metric, pd.Series(index=audit_summary.index)).ge(THRESHOLDS[metric])

    pass_columns = [f"{metric}_pass" for metric in metric_columns]
    audit_summary["provisional_pass"] = (
        audit_summary["season_found"].fillna(False)
        & audit_summary[pass_columns].fillna(False).all(axis=1)
    )

    display_columns = [
        "league", "season", "fd_fixtures", "api_fixtures",
        "fixture_coverage", "football_data_join", "both_lineups",
        "twenty_two_starters", "player_stats_matches",
        "starter_minutes", "player_identity", "xg_flag_rate",
        "provisional_pass",
    ]
    display(audit_summary[[column for column in display_columns if column in audit_summary]].style.format({
        column: "{:.1%}" for column in display_columns if column in THRESHOLDS or column == "xg_flag_rate"
    }))

No results yet. Run the previous cell with RUN_AUDIT = True.


In [29]:
if detail_rows:
    detail_df = pd.DataFrame(detail_rows)

    field_rows = []
    for (league, season), group in detail_df.groupby(["league", "season"]):
        field_sets = [set(fields) for fields in group["player_stat_fields"] if fields]
        fields = sorted(set().union(*field_sets)) if field_sets else []
        field_rows.append({"league": league, "season": season, "player_stat_fields": ", ".join(fields)})
    field_summary = pd.DataFrame(field_rows)
    display(field_summary)

    errors = detail_df[detail_df["lineup_error"].notna() | detail_df["stats_error"].notna()][
        ["league", "season", "match_id", "lineup_error", "stats_error"]
    ]
    if not errors.empty:
        print("Endpoint errors to inspect:")
        display(errors)

if unmatched_rows:
    print("Unmatched provider names. Add only confirmed differences to TEAM_ALIASES, then rerun; cached API calls will be reused.")
    display(pd.DataFrame(unmatched_rows).drop_duplicates().head(100))

In [30]:
# Small, reusable audit outputs. This does not save raw player data.
if audit_rows:
    output_dir = Path("artifacts/statsapi_coverage_audit")
    output_dir.mkdir(parents=True, exist_ok=True)
    audit_summary.to_csv(output_dir / "league_season_summary.csv", index=False)
    if detail_rows:
        detail_df.to_csv(output_dir / "sample_match_details.csv", index=False)
        field_summary.to_csv(output_dir / "player_stat_fields.csv", index=False)
    if unmatched_rows:
        pd.DataFrame(unmatched_rows).drop_duplicates().to_csv(output_dir / "unmatched_teams.csv", index=False)
    print("Saved audit tables to", output_dir)

In [ ]:
api_get("/football/matches/mt_138510863/player-stats")